In [1]:
!pip install -q transformers accelerate bitsandbytes pypdf

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch, re, json
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 7.9 MB/s eta 0:00:00


In [2]:
PDF_PATH = Path("RAG_base/rag_base.pdf")
OUT_PATH = Path("domain_qa.jsonl")

# === Load small multilingual model ===
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

# === PDF -> text ===
def pdf_to_text(path):
    reader = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in reader.pages)

def clean_text(t):
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)
    return re.sub(r"\s+\n", "\n", t).strip()

text = clean_text(pdf_to_text(PDF_PATH))

# Split into chapters (simple heuristic)
chapters = re.split(r"(?i)\n\s*(?:Kapitel|Chapter)\s+\d+", text)
chapters = [c.strip() for c in chapters if len(c.split()) > 80]  # skip tiny bits

print("Detected chapters:", len(chapters))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'RAG_base/rag_base.pdf'

In [ ]:
def gen_qa(text, n=3):
    prompt = f"""Erstelle {n} sinnvolle Frage-Antwort-Paare (Q&A) auf Deutsch aus folgendem Abschnitt.
Die Fragen sollen prüfend, erklärend oder definierend sein. Gib sie im JSON-Format als Liste zurück.

Abschnitt:
{text[:2000]}  # kürze für Sicherheit
"""
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=512, temperature=0.7)
    decoded = tok.decode(out[0], skip_special_tokens=True)
    # crude attempt to extract JSON
    m = re.search(r"\[.*\]", decoded, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except:
            pass
    return []

qas = []
for i, ch in enumerate(chapters[:10], 1):  # adjust slice for more
    qas += gen_qa(ch, n=3)
    print(f"Chapter {i}: got {len(qas)} QAs total")

with OUT_PATH.open("w", encoding="utf-8") as f:
    for qa in qas:
        if "question" in qa and "answer" in qa:
            f.write(json.dumps(qa, ensure_ascii=False) + "\n")

print("✅ Wrote", OUT_PATH)
